# 📘 Bloque 1: Fundamentos de Machine Learning

**Objetivo:** Entender los tipos de aprendizaje, los algoritmos clásicos y cómo evaluar modelos correctamente.

---

## 1. ¿Qué es Machine Learning?

Machine Learning (ML) es la rama de la IA que permite a los sistemas **aprender patrones a partir de datos** sin ser programados explícitamente para cada caso.

### Tipos de aprendizaje:

| Tipo | Descripción | Ejemplo |
|---|---|---|
| **Supervisado** | Aprende con datos etiquetados (X → y) | Clasificar emails como spam o no spam |
| **No supervisado** | Encuentra patrones sin etiquetas | Agrupar clientes por comportamiento |
| **Por refuerzo** | Un agente aprende por recompensas/penalizaciones | AlphaGo, coches autónomos |

---

## 2. Pipeline básico de ML con scikit-learn

El flujo estándar de cualquier proyecto de ML es siempre el mismo:

### 2.1. Importar las librerias

#### 2.1.1. Librerías utilizadas

Estas 4 librerias son la base de cualquier proyecto de Data Science en Python.

In [2]:
import numpy as np  # Operaciones matemáticas y arrays
import pandas as pd  # Tablas de datos (DataFrames)
import matplotlib.pyplot as plt  # Gráficas
import seaborn as sns  # Gráficas más bonitas sobre matplotlib



**numpy** — el motor matemático
```python
import numpy as np
```
Trabaja con arrays y matrices de números de forma muy eficiente. Es la base de todo lo demás — pandas, scikit-learn y PyTorch usan numpy por debajo. Sin numpy no existe el ML en Python.

---

**pandas** — tablas de datos
```python
import pandas as pd
```
Es el Excel de Python. Te permite cargar CSVs, filtrar filas, limpiar datos, hacer agrupaciones... El 80% del trabajo de un Data Engineer/Scientist es manipular datos con pandas.

---

**matplotlib** — gráficas
```python
import matplotlib.pyplot as plt
```
La librería base para hacer gráficas en Python. Líneas, barras, histogramas, scatter plots... Es muy flexible pero requiere bastante código para que queden bien.

---

**seaborn** — gráficas más bonitas
```python
import seaborn as sns
```
Construida encima de matplotlib. Con menos código produces gráficas más estéticas y útiles para análisis estadístico. Muy usada para explorar datos antes de modelar.


#### 2.1.2. Librerías de scikit-learn

scikit-learn (sklearn) es la librería de Machine Learning clásico más usada en Python. Incluye todo lo necesario para construir un pipeline de ML completo: algoritmos de clasificación, regresión y clustering, herramientas para dividir y preprocesar datos, y métricas para evaluar modelos. Es el punto de partida estándar antes de pasar a Deep Learning con PyTorch.

In [3]:
from sklearn.datasets import load_iris  # Dataset de ejemplo             
from sklearn.model_selection import train_test_split  # Dividir datos
from sklearn.preprocessing import StandardScaler  # Escalar features
from sklearn.metrics import classification_report, confusion_matrix  # Evaluar modelo

**sklearn.datasets — load_iris**
```python
from sklearn.datasets import load_iris
```
Carga datasets de ejemplo que vienen incluidos en scikit-learn. Iris es el más clásico — 150 flores con 4 medidas cada una y 3 especies posibles. Lo usamos para practicar sin necesidad de buscar datos reales.

---

**sklearn.model_selection — train_test_split**
```python
from sklearn.model_selection import train_test_split
```
Divide tu dataset en dos partes: una para entrenar el modelo y otra para evaluarlo. Es fundamental porque si entrenas y evalúas con los mismos datos, el modelo hace trampa — memoriza en lugar de aprender.

---

**sklearn.preprocessing — StandardScaler**
```python
from sklearn.preprocessing import StandardScaler
```
Escala los números para que todas las features estén en el mismo rango. Por ejemplo, si una columna va de 0 a 1 y otra de 0 a 1.000.000, el modelo le da más importancia a la segunda solo por ser más grande. El scaler lo corrige poniendo todo con media=0 y desviación=1.

---

**sklearn.metrics — classification_report, confusion_matrix**
```python
from sklearn.metrics import classification_report, confusion_matrix
```
Herramientas para medir qué tan bien funciona tu modelo una vez entrenado.
- `classification_report` → te da precision, recall y F1 por cada clase en una tabla
- `confusion_matrix` → muestra una matriz con los aciertos y errores del modelo por clase

### 2.2. Cargar el dataset Iris

```python
iris = load_iris()
X, y = iris.data, iris.target
```

Iris es el "Hello World" del ML. Contiene medidas de 150 flores de 3 especies distintas.

- `X` → las features (lo que el modelo usa para aprender): longitud/anchura del sépalo y pétalo
- `y` → el target (lo que queremos predecir): la especie (0, 1 o 2)

In [ ]:
# Cargar datos
iris = load_iris()
X, y = iris.data, iris.target

#### Los prints

```python
print(f"Shape de X: {X.shape}")  # -> (150, 4) = 150 filas, 4 columnas
print(f"Clases: {iris.target_names}")  # -> ['setosa', 'versicolor', 'virginica']
print(f"Primeras filas:\n{pd.DataFrame(X, columns=iris.feature_names).head()}")
```

El último print convierte el array de numpy en un DataFrame de pandas para verlo como una tabla legible. `.head()` muestra solo las primeras 5 filas.

In [5]:
print(f"Shape de X: {X.shape}")  # (150, 4) -> 150 muestras/filas, 4 features/columnas
print(f"Clases: {iris.target_names}")  # ['setosa', 'versicolor', 'virginica']
print(f"Primeras filas:\n{pd.DataFrame(X, columns=iris.feature_names).head()}")

Shape de X: (150, 4)
Clases: ['setosa' 'versicolor' 'virginica']
Primeras filas:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2


### 2.3. Split train/test

Divide los datos en dos grupos: uno para entrenar el modelo y otro para evaluarlo.
Si usáramos todos los datos para entrenar y luego evaluáramos con los mismos, el modelo haría trampa — memoriza en lugar de aprender.

**Parámetros:**

- `test_size=0.2` → 20% de los datos van a test, 80% a train (150 flores → 120 train, 30 test)
- `random_state=42` → fija la aleatoriedad para que el split sea siempre el mismo. El 42 es convención, puedes poner cualquier número
- `stratify=y` → garantiza que la proporción de cada especie sea la misma en train y en test. Sin esto podría pasar que todas las flores de una especie acabaran en train y el modelo nunca la viera en test

**Resultado:**

- `X_train` → features de entrenamiento (120 filas x 4 columnas)
- `X_test` → features de evaluación (30 filas x 4 columnas)
- `y_train` → especies de entrenamiento (120 valores)
- `y_test` → especies de evaluación (30 valores)

In [ ]:
# Split train/test (regla general: 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

### 2.3. Escalado de features

Escala los datos para que todas las features estén en el mismo rango. Muchos algoritmos de ML son sensibles a la escala de los números — si una columna va de 0 a 1 y otra de 0 a 1.000.000, el modelo le da más peso a la segunda simplemente por ser más grande, no porque sea más informativa.

El `StandardScaler` transforma cada columna para que tenga **media=0 y desviación típica=1**.

**Parámetros:**

- `fit_transform(X_train)` → aprende la media y desviación de los datos de train, y los transforma. Siempre sobre train
- `transform(X_test)` → aplica la misma transformación aprendida en train sobre test. **Nunca se hace `fit` sobre test** — si no, estarías usando información del futuro para escalar, lo cual falsea la evaluación

**Resultado:**

- `X_train_scaled` → datos de entrenamiento escalados
- `X_test_scaled` → datos de test escalados con la misma escala que train

In [ ]:
# Escalado (muy importante para muchos algoritmos)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit+transform en train
X_test_scaled = scaler.transform(X_test)        # solo transform en test (¡nunca fit!)

## 3. Algoritmos clásicos

### 3.1 Regresión Logística (clasificación)
> A pesar del nombre, es un clasificador. Aprende una frontera lineal entre clases.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)

print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=iris.target_names))

### 3.2 Random Forest (árbol de decisión con ensemble)
> Crea muchos árboles y combina sus predicciones. Robusto y fácil de usar.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=iris.target_names))

# Importancia de features
feat_importance = pd.Series(rf_model.feature_importances_, index=iris.feature_names)
feat_importance.sort_values().plot(kind='barh', title='Importancia de Features')
plt.tight_layout()
plt.show()

### 3.3 SVM — Support Vector Machine
> Encuentra el hiperplano que maximiza el margen entre clases. Muy efectivo en espacios de alta dimensión.

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

print("=== SVM ===")
print(classification_report(y_test, y_pred_svm, target_names=iris.target_names))

## 4. Evaluación de modelos

### Métricas principales para clasificación:

| Métrica | Fórmula | Cuándo usarla |
|---|---|---|
| **Accuracy** | TP+TN / total | Clases balanceadas |
| **Precision** | TP / (TP+FP) | Minimizar falsos positivos |
| **Recall** | TP / (TP+FN) | Minimizar falsos negativos (ej: enfermedades) |
| **F1-Score** | 2·(P·R)/(P+R) | Balance entre precision y recall |
| **ROC-AUC** | Área bajo curva ROC | Comparar modelos con clases desbalanceadas |

In [ ]:
# Matriz de confusión visual
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.title('Matriz de Confusión — Random Forest')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

## 5. Overfitting vs Underfitting

> Este es uno de los conceptos más importantes en ML.

- **Underfitting**: El modelo es demasiado simple, no aprende ni en train ni en test.
- **Overfitting**: El modelo memoriza train pero falla en test (no generaliza).
- **Buen modelo**: Buena métrica tanto en train como en test.

**Solución al overfitting**: más datos, regularización (L1/L2), dropout, cross-validation.

In [ ]:
from sklearn.model_selection import cross_val_score

# Cross-validation de 5 folds: entrena 5 veces con distintos splits
scores = cross_val_score(rf_model, X, y, cv=5, scoring='f1_macro')

print(f"F1 por fold: {scores.round(3)}")
print(f"F1 medio: {scores.mean():.3f} ± {scores.std():.3f}")

## 6. Comparativa de modelos

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

modelos = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf'),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)
    f1 = f1_score(y_test, y_pred, average='macro')
    resultados[nombre] = f1

pd.Series(resultados).sort_values().plot(
    kind='barh', title='F1-Score por Modelo', xlim=(0.8, 1.0), color='steelblue'
)
plt.tight_layout()
plt.show()

---

## ✅ Resumen del bloque

- Aprendiste los **3 tipos de ML**: supervisado, no supervisado, refuerzo
- Implementaste un **pipeline completo**: carga → split → escalar → entrenar → evaluar
- Conoces los **algoritmos clásicos**: Regresión Logística, Random Forest, SVM, KNN
- Entiendes las **métricas**: Accuracy, Precision, Recall, F1, matriz de confusión
- Sabes qué es **overfitting** y cómo detectarlo con cross-validation

---

## ➡️ Siguiente paso

Continúa con el **Bloque 2: Deep Learning con PyTorch** → `02_deep_learning_pytorch.ipynb`